In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
!export OMP_NUM_THREADS=4

In [2]:
model_path ='../model/Qwen2.5-Math-1.5B'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    attn_implementation='flash_attention_2',
    trust_remote_code=True,
).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.


In [3]:
prompt = 'Once upon a time'
input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
output = model(input_ids)
output

CausalLMOutputWithPast(loss=None, logits=tensor([[[11.5000,  7.9062,  2.0938,  ..., -1.5938, -1.5938, -1.5938],
         [ 3.7188,  4.2188, -2.3750,  ..., -2.2500, -2.2500, -2.2500],
         [ 1.1484, -0.4688,  0.0942,  ..., -2.9844, -2.9844, -2.9844],
         [ 4.5625,  2.8906, -2.7500,  ..., -3.4531, -3.4531, -3.4531]]],
       device='cuda:0', dtype=torch.bfloat16, grad_fn=<UnsafeViewBackward0>), past_key_values=<transformers.cache_utils.DynamicCache object at 0x7f6573d941d0>, hidden_states=None, attentions=None)

In [4]:
output.logits

tensor([[[11.5000,  7.9062,  2.0938,  ..., -1.5938, -1.5938, -1.5938],
         [ 3.7188,  4.2188, -2.3750,  ..., -2.2500, -2.2500, -2.2500],
         [ 1.1484, -0.4688,  0.0942,  ..., -2.9844, -2.9844, -2.9844],
         [ 4.5625,  2.8906, -2.7500,  ..., -3.4531, -3.4531, -3.4531]]],
       device='cuda:0', dtype=torch.bfloat16, grad_fn=<UnsafeViewBackward0>)

# tokenize_prompt_and_output
+ Qwen的pad.id不一定为0

In [5]:
tokenizer.pad_token_id
tokenizer([prompt+prompt,prompt], padding=True)

{'input_ids': [[12522, 5193, 264, 882, 12522, 5193, 264, 882], [12522, 5193, 264, 882, 151643, 151643, 151643, 151643]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 0, 0, 0, 0]]}

In [6]:
# example
prompt_strs = ['What is 1+1?','What is the capital of France?']
output_strs = [' 2',' Paris']

prompt_ids = tokenizer(prompt_strs,padding=False).input_ids # only ids: b l1
output_ids = tokenizer(output_strs,padding=False).input_ids# 

# 拼接后的prompt+output,但是长度不一
input_ids = []
prompt_masks = []
for p,o in zip(prompt_ids,output_ids):
    prompt_masks.append([1]*len(p)+[0]*len(o))
    input_ids.append(p+o)
max_length = max([len(ids) for ids in input_ids])
attention_masks = []

for i in range(len(input_ids)):
    attention_masks.append([1]*len(input_ids[i])+[0]*(max_length-len(input_ids[i])))
    prompt_masks[i] = prompt_masks[i]+[0]*(max_length-len(prompt_masks[i]))
    input_ids[i] = input_ids[i] + [0]*(max_length-len(input_ids[i]))

attention_masks = torch.tensor(attention_masks)
prompt_masks = torch.tensor(prompt_masks)
input_ids = torch.tensor(input_ids)
mask = (attention_masks | prompt_masks).bool()
labels = input_ids[:,1:]
input_ids.shape,labels.shape,attention_masks.shape,prompt_masks.shape,mask.shape
# mask


# # input_ids : b l
# inputs = tokenizer(prompt_and_output_strs,return_tensors='pt',padding=True)# 批次中最长序列的长度
# input_ids = inputs.input_ids
# labels = input_ids[:,1:]

# # padding_mask : b l
# padding_mask = inputs.attention_mask

# # prompt mask: b ? -> b l 
# max_length = padding_mask.shape[1]
# for i in range(len(prompt_masks)):
#     prompt_masks[i] = prompt_masks[i]+[0]*(max_length-len(prompt_masks[i]))
# prompt_masks = torch.tensor(prompt_masks)

# attention_mask = (padding_mask | prompt_masks).bool()


# print('input_ids shape:', input_ids.shape)
# print('labels shape:', labels.shape)    
# print('padding_mask shape:', padding_mask.shape)
# print('prompt_masks shape:', prompt_masks.shape)
# print('attention_mask shape:', attention_mask.shape)



(torch.Size([2, 9]),
 torch.Size([2, 8]),
 torch.Size([2, 9]),
 torch.Size([2, 9]),
 torch.Size([2, 9]))

In [7]:
def tokenize_prompt_and_output(prompt_strs, output_strs, tokenizer):
    """
    对提示和输出字符串进行分词,并构建一个掩码,标记响应 token（值为 1）,其余（提示或填充）为 0。

    Args:
        prompt_strs: List[str] —— 提示字符串列表。
        output_strs: List[str] —— 输出字符串列表。
        tokenizer: PreTrainedTokenizer —— 用于分词的分词器。

    Returns:
        dict[str, torch.Tensor]:
            设 prompt_and_output_lens 为各拼接后序列的长度列表,
            返回字典包含以下键:
            - input_ids: shape (batch_size, max(prompt_and_output_lens) - 1)
                         拼接后的 token 序列（去掉最后一个 token）
            - labels: shape 同 input_ids,为 input_ids 右移一位（即去掉第一个 token）
            - response_mask: shape 同 input_ids,响应 token 对应位置为 True,其余为 False
    """

    # 填充要填充指定的pad_id,掩码中1代表有用的token,0代表无用的token
    prompt_ids = tokenizer(prompt_strs, 
        add_special_tokens=False,
        padding=False,
        truncation=False,
        return_attention_mask=False,).input_ids
    output_ids = tokenizer(output_strs,
        add_special_tokens=False,
        padding=False,
        truncation=False,
        return_attention_mask=False,).input_ids
    input_ids = []
    prompt_masks = []

    # 拼接输入和输出,获得prompt_masks
    for p, o in zip(prompt_ids, output_ids):
        input_ids.append(p + o)
        prompt_masks.append([0] * len(p) + [1] * len(o))

    # 获取最长长度
    max_length = max([len(ids) for ids in input_ids])

    # 获取padding_mask b ? -> b l
    padding_masks = []
    pad_id = tokenizer.pad_token_id
    for i in range(len(input_ids)):
        # [1,1,0]
        padding_masks.append([1] * len(input_ids[i]) + [0] * (max_length - len(input_ids[i])))
        # [0 1 0]
        prompt_masks[i] = prompt_masks[i] + [0] * (max_length - len(prompt_masks[i]))
        # [x x y]
        input_ids[i] = input_ids[i] + [pad_id] * (max_length - len(input_ids[i]))
    
    padding_masks = torch.tensor(padding_masks)
    prompt_masks = torch.tensor(prompt_masks)
    input_ids = torch.tensor(input_ids)
    mask = (padding_masks & prompt_masks).bool()
    return {
        'input_ids': input_ids[:,:-1],# 去掉最后一个token(no label)
        'labels': input_ids[:, 1:].clone(), 
        'response_mask': mask[:,1:]
    }

In [8]:
prompt_strs = ['Hello, world!', 'This is a test.', 'This is another test.']
output_strs = ['Hello, world!', 'This is a test.', 'This is another test.']

In [9]:

res = tokenize_prompt_and_output(
    prompt_strs = prompt_strs,
    output_strs = output_strs,
    tokenizer = tokenizer
)
"""
tensor([[False, False, False, False,  True,  True,  True,  True, False, False],
        [False, False, False, False, False,  True,  True,  True,  True,  True],
        [False, False, False, False, False,  True,  True,  True,  True,  True]])
"""
res['response_mask']


tensor([[False, False, False,  True,  True,  True,  True, False, False],
        [False, False, False, False,  True,  True,  True,  True,  True],
        [False, False, False, False,  True,  True,  True,  True,  True]])

逐token熵记录 在强化学习（RL）过程中,记录逐token熵通常很有用,可用于观察模型的预测分布是否变得（过度）自信。现在我们将实现这一功能,并比较各种微调方法对模型预测熵的影响。
给定 SFT 或 RL 模型的 logits,我们将计算每个 token 的熵,即每个下一个 token 预测的熵。
问题（compute_entropy）:Per-token entropy (1 point)

In [10]:
def softmax(
    logits:torch.Tensor,
    dim:int =-1,
):
    m = logits.max(dim=dim,keepdim=True).values
    exp_logits = torch.exp(logits - m)
    return exp_logits / exp_logits.sum(dim=dim, keepdim=True)

In [11]:
def compute_entropy(logits: torch.Tensor) -> torch.Tensor:
	"""
	功能:获取下一个 token 预测的熵（即在词汇表维度上的熵）。
	
	参数:
	- logits: torch.Tensor,形状为 (batch_size, sequence_length, vocab_size),包含未归一化的 logits。
	
	返回值:
	- torch.Tensor,形状为 (batch_size, sequence_length),表示每个下一个 token 预测的熵。
	"""
	probs = softmax(logits,dim=-1)
	log_probs = torch.log(probs + 1e-6)
	entropy = - log_probs * probs
	return entropy.sum(dim=-1)

In [12]:
x = torch.randn((2,3,4))
compute_entropy(x).shape

torch.Size([2, 3])

从模型获取log-probabilities 从模型中获取对数概率是监督微调（SFT）和强化学习（RL）中都需要用到的基础操作。

In [13]:
def get_response_log_probs(
    model: AutoModelForCausalLM,
    input_ids: torch.Tensor,
    labels: torch.Tensor,
    return_token_entropy: bool = False,
) -> dict[str, torch.Tensor]:
    """参数:
    - model:PreTrainedModel,用于评分的HuggingFace模型（若无需计算梯度,需放置在正确设备上并处于推理模式）。
    - input_ids:torch.Tensor,形状为（batch_size, sequence_length）,由分词方法生成的拼接后的提示词+响应token。
    - labels:torch.Tensor,形状为（batch_size, sequence_length）,由分词方法生成的标签。
    - return_token_entropy:bool,若为True,通过调用`compute_entropy`额外返回逐token熵。

    返回值:
    - dict[str, torch.Tensor]:
    - "log_probs":形状为（batch_size, sequence_length）,条件对数概率\(log p_{\theta}(x_t | x_{<<t})\)。
    - "token_entropy"（可选）:形状为（batch_size, sequence_length）,每个位置的逐token熵（仅当return_token_entropy=True时存在）。
    """
    logits = model(input_ids).logits  # b l v
    log_probs_all = torch.log_softmax(logits, dim=-1)  # b l v
    
    log_probs = torch.gather(
        log_probs_all, 
        dim=-1, 
        index=labels.unsqueeze(-1)
    ).squeeze(-1)  # b l
    
    if return_token_entropy:
        entropy = compute_entropy(logits)  # b l
        return {
            "log_probs": log_probs,
            "token_entropy": entropy,
        }
    else:
        return {
            "log_probs": log_probs,
            "token_entropy": None,
        }

<>:15: SyntaxWarning: invalid escape sequence '\('
<>:15: SyntaxWarning: invalid escape sequence '\('
/tmp/ipykernel_2536/2236930521.py:15: SyntaxWarning: invalid escape sequence '\('
  - "log_probs":形状为（batch_size, sequence_length）,条件对数概率\(log p_{\theta}(x_t | x_{<<t})\)。


问题（masked_normalize）：掩码归一化（1分）
+ 交付要求：实现masked_normalize方法，在考虑布尔掩码的前提下，对张量元素求和并通过常数进行归一化。

In [ ]:
def masked_normalize(
    tensor: torch.Tensor,
    mask: torch.Tensor,
    normalize_constant: float,
    dim: int | None = None,
) -> torch.Tensor:
    """
    对指定维度求和并通过常数归一化，仅考虑掩码中值为1的元素。

    参数：
    - tensor：torch.Tensor，需求和并归一化的张量。
    - mask：torch.Tensor，与tensor形状相同；值为1的位置会被纳入求和范围。
    - normalize_constant：float，用于归一化的除数常数。
    - dim：int | None，归一化前要求和的维度；若为None，对所有维度求和。

    返回值：
    - torch.Tensor，归一化后的和，其中掩码元素（mask == 0）不参与求和。
    """
    masked_tensor = tensor * mask
    summed = masked_tensor.sum(dim=dim,keepdim=True)
    norm = summed / normalize_constant
    return norm.sum(dim=dim)

问题（sft_microbatch_train_step）：Microbatch train step (3 points)
交付要求
+ 实现监督微调（SFT）的单个微批次更新，包括交叉熵损失计算、掩码求和及梯度缩放。

In [ ]:
def sft_microbatch_train_step(
    policy_log_probs: torch.Tensor,
    response_mask: torch.Tensor,
    gradient_accumulation_steps: int,
    normalize_constant: float = 1.0,
) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
    """
    对微批次执行前向传播和反向传播。
	
	参数：
	- policy_log_probs：形状为（batch_size, sequence_length），来自待训练监督微调（SFT）策略的逐token对数概率。
	- response_mask：形状为（batch_size, sequence_length），响应token对应位置为1，提示词/填充token对应位置为0。
	- gradient_accumulation_steps：每个优化器步骤对应的微批次数量。
	- normalize_constant：用于除法归一化的常数，默认设为1.0即可。
	
	返回值：
	- tuple[torch.Tensor, dict[str, torch.Tensor]]：
	  - loss：标量张量，微批次损失（已根据梯度累积进行调整），返回该值用于日志记录。
	  - metadata：字典，包含底层损失调用的元数据及其他需记录的统计信息。
    """
    # SFT的目标函数:-log(y|x)

    loss = - masked_normalize(
        tensor = policy_log_probs,
        mask = response_mask,
        normalize_constant = normalize_constant,
        dim=-1,
    ).mean()
    scaled_loss = loss / gradient_accumulation_steps
    scaled_loss.backward()
    
    return scaled_loss,{
        'loss': scaled_loss,
        'unscaled_loss': loss,
    }


Logging generations in-the-loop
在模型训练循环中记录生成结果是良好的实践，监督微调（SFT）/强化学习（RL）场景也不例外。
编写log_generations函数，用于让模型对给定提示词（如从验证集中采样的提示词）生成响应并记录日志。建议为每个示例至少记录以下内容：

+ 输入提示词。
+ 监督微调（SFT）/强化学习（RL）模型生成的响应。
+ 真实答案。
+ 奖励信息，包括格式、答案及总奖励。
+ 响应的平均token熵。
+ 平均响应长度、正确响应的平均长度及错误响应的平均长度。